In [1]:
import numpy as np
import pandas as pd

import plotly.express as px

data = np.load('../data/metr_la_new.npz', allow_pickle=True)
list(data.keys())

dataset = data['targets']
dataset_size = len(dataset)

for i in range(0, len(dataset)):
    for j in range(0, len(dataset[i])):
        if np.isnan(dataset[i][j]) and i == 0:
            dataset[i][j] = 0
        if np.isnan(dataset[i][j]):
            dataset[i][j] = dataset[max(i - 1, 0)][j]


In [2]:
train_size = int(0.6 * dataset_size)
test_size =  int(0.2 * dataset_size)
def dataset_for_vertice(vertice):
    return dataset[:, vertice]
coef = 6
pred_cnt = 12

In [3]:
import random
samples_train = [i for i in random.sample(range(coef, int(dataset_size * 0.7)), train_size)]

X_train = [[dataset_for_vertice(j)[i - coef: i] for j in range(0, 207)] for i in samples_train]
X_train = np.array(X_train)
#X_train = X_train.reshape(X_train.shape[0], X_train.shape[1] * X_train.shape[2])

y_train = [[dataset_for_vertice(j)[i:i + pred_cnt] for j in range(0,207)] for i in samples_train]
y_train = np.array(y_train)
y_train = y_train.reshape(y_train.shape[0], y_train.shape[1] * y_train.shape[2])

samples_test = [i for i in random.sample(range(int(dataset_size * 0.7), dataset_size - pred_cnt), test_size)]
X_test = [[dataset_for_vertice(j)[i - coef: i] for j in range(0, 207)] for i in samples_test]
X_test = np.array(X_test)
#X_test = X_test.reshape(X_test.shape[0], X_test.shape[1] * X_test.shape[2])

y_test = [[dataset_for_vertice(j)[i:i + pred_cnt] for j in range(0,207)] for i in samples_test]
y_test = np.array(y_test)
y_test = y_test.reshape(y_test.shape[0], y_test.shape[1] * y_test.shape[2])

In [4]:
X_train.shape

(20563, 207, 6)

In [5]:
from features import create_features, normalize
graph = data['edges_pruned_by_partial_correlation']
graph_view = np.zeros((207, 207))

for node_1, node_2 in graph:
    graph_view[node_2][node_1] = 1
    

In [6]:
X_train_with_features = create_features(
                          number_of_timestamps = X_train.shape[0],
                          graph = graph_view,
                          nodes_number=207,
                          features = np.array(X_train),
                          modes = ['min', 'max']
                         )
X_test_with_features = create_features(
                          number_of_timestamps = X_test.shape[0],
                          graph = graph_view,
                          nodes_number=207,
                          features = np.array(X_test),
                          modes = ['min', 'max']
                         )

100%|██████████| 6854/6854 [00:20<00:00, 340.72it/s]


In [7]:
X_train_normal_features = normalize(X_train_with_features)
X_tr = np.concatenate((X_train, X_train_normal_features), axis= 2)
X_test_normal_features = normalize(X_test_with_features)
X_te = np.concatenate((X_test, X_test_normal_features), axis= 2)

print(X_tr.shape, X_te.shape)

[[[50.74931791 50.74733386 50.75139064 50.75453369 50.75272807
   50.75606183 58.85213104 58.85170064 58.85344841 58.85606798
   58.85541646 58.85610506]]]
[[[19.45487129 19.45452213 19.45381972 19.45385002 19.45351149
   19.45382293 18.36410843 18.36337992 18.36326578 18.36214878
   18.36321213 18.36298366]]]
[[[19.45487129 19.45452213 19.45381972 19.45385002 19.45351149
   19.45382293 18.36410843 18.36337992 18.36326578 18.36214878
   18.36321213 18.36298366]]]
[[[50.45382193 50.45827603 50.46125885 50.46498365 50.45977908
   50.47512102 58.63249914 58.63552799 58.63961722 58.63813154
   58.63691405 58.64273618]]]
[[[19.67375262 19.66906335 19.66770427 19.66953848 19.66414835
   19.6620657  18.55830328 18.5550815  18.55510828 18.55634625
   18.55467942 18.55347406]]]
[[[19.67375262 19.66906335 19.66770427 19.66953848 19.66414835
   19.6620657  18.55830328 18.5550815  18.55510828 18.55634625
   18.55467942 18.55347406]]]
(20563, 207, 18) (6854, 207, 18)


In [8]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_tr = torch.tensor(X_tr.reshape(X_tr.shape[0], X_tr.shape[1] * X_tr.shape[2]), dtype=float).to(device)
y_train = torch.tensor(y_train, dtype=float).to(device)

X_te = torch.tensor(X_te.reshape(X_te.shape[0], X_te.shape[1] * X_te.shape[2]), dtype=float).to(device)
y_test = torch.tensor(y_test, dtype=float).to(device)

tensor_data = TensorDataset(X_tr.float(), y_train.float())

dataloader_train = DataLoader(tensor_data, shuffle = True, batch_size = 16)

In [9]:
X_tr.shape

torch.Size([20563, 3726])

In [10]:
model = nn.Linear(207 * 18, 2484)

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr = 0.03)
loss_fn = nn.MSELoss()

print(y_train.shape)

torch.Size([20563, 2484])


In [12]:
from tqdm import tqdm
epochs = 100
losses = []
for i in tqdm(range(epochs)):
    for x, y in dataloader_train:
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

  0%|          | 0/100 [00:00<?, ?it/s]


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument mat1 in method wrapper_CUDA_addmm)

In [ ]:
px.line(y = losses)

## Independent linear model for each node

In [7]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn

class LinearEnsemble(nn.Module):
    def __init__(self,
                 dataset_size: int,
                 n_input_features: int,
                 n_output_features: int,
                 edges: np.ndarray = None,  # omitted in this model
                 graph: np.ndarray = None,
                 shared_weights: bool = False,
                 ):
        super().__init__()
        self._n_independent_models = dataset_size if not shared_weights else 1

        self.W = nn.Parameter(
            data=torch.randn(size=(self._n_independent_models, n_output_features, n_input_features)),
            requires_grad=True,
        )

        self.bias = nn.Parameter(
            data=torch.zeros(self._n_independent_models, n_output_features),
            requires_grad=True,
        )

    def forward(self, X):
        hadamard_product = self.W * X.unsqueeze(2)  # emulates linear layer independent for every node
        
        return hadamard_product.sum(-1) + self.bias  # reduce (aggregate) element-wise products and sum with bias

In [8]:
X_train_for_linear_ensemble = np.concatenate((X_train, X_train_with_features), axis= 2)
X_test_for_linear_ensemble = np.concatenate((X_test, X_test_with_features), axis= 2)


In [9]:
X_train_for_linear_ensemble_torch = torch.from_numpy(normalize(X_train_for_linear_ensemble)).float()
X_test_for_linear_ensemble_torch = torch.from_numpy(normalize(X_test_for_linear_ensemble)).float()

Y_train_ensemble = torch.tensor([[dataset_for_vertice(j)[i:i + pred_cnt] for j in range(0,207)] for i in samples_train]).float()
Y_test_ensemble = torch.tensor([[dataset_for_vertice(j)[i:i + pred_cnt] for j in range(0,207)] for i in samples_test]).float()


ensemble_dataset_train = TensorDataset(X_train_for_linear_ensemble_torch, Y_train_ensemble)
ensemble_dataset_test = TensorDataset(X_test_for_linear_ensemble_torch, Y_test_ensemble)

[[[58.4629367  58.4633984  58.4625932  58.46187977 58.45924684
   58.45703946 50.81552273 50.81774434 50.81619499 50.81522127
   50.81127196 50.80886084 58.88347287 58.88346461 58.88347246
   58.88357787 58.8815173  58.88040892]]]
[[[13.06689824 13.06744506 13.06800432 13.06926346 13.07317692
   13.07588926 19.42260494 19.42292673 19.4230455  19.42374317
   19.4256511  19.42748628 18.34429114 18.34492238 18.34466797
   18.3451388  18.34668625 18.34723711]]]
[[[13.06689824 13.06744506 13.06800432 13.06926346 13.07317692
   13.07588926 19.42260494 19.42292673 19.4230455  19.42374317
   19.4256511  19.42748628 18.34429114 18.34492238 18.34466797
   18.3451388  18.34668625 18.34723711]]]
[[[58.17905258 58.17846802 58.17805707 58.18021658 58.18328283
   58.18361387 50.41679969 50.41311125 50.41277395 50.41633726
   50.41578098 50.41368386 58.59170638 58.59227338 58.58977222
   58.5934859  58.59487012 58.59517988]]]
[[[13.49274991 13.49206096 13.49050429 13.49446915 13.49012438
   13.4869706

/var/tmp/ipykernel_310955/3621475539.py:4: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1720538456841/work/torch/csrc/utils/tensor_new.cpp:278.)
  Y_train_ensemble = torch.tensor([[dataset_for_vertice(j)[i:i + pred_cnt] for j in range(0,207)] for i in samples_train]).float()


In [13]:
dataloader_train_ensemble = DataLoader(ensemble_dataset_train, shuffle = True, batch_size = 1024 * 2 ** 4)
dataloader_test_ensemble = DataLoader(ensemble_dataset_test, shuffle = False, batch_size = 1024 * 2 ** 4)

In [15]:
from tqdm import tqdm

device = "cuda:3"

In [16]:
from torch.amp import GradScaler, autocast

In [154]:
def validate(model, val_loader, loss_fn):
    model.eval()

    losses = []
    for x, y in val_loader:
        with autocast(device, dtype=torch.float16):
            y_pred = model(x.to(device))
            loss = loss_fn(y_pred, y.to(device))

        losses.append(loss.detach())
    losses = np.mean(torch.tensor(losses).cpu().numpy().tolist())
    return losses

def train_and_validate(model, train_loader, val_loader, lr = 0.1, n_epochs: int = 500, weight_decay: float = 1e-6):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    loss_fn = nn.L1Loss()
    grad_scaler = GradScaler(device=device)

    losses = []
    for i in tqdm(range(n_epochs)):
        for x, y in train_loader:
            with autocast(device, dtype=torch.float16):
                y_pred = model(x.to(device))
                # print(y_pred.shape, y.shape)
                loss = loss_fn(y_pred, y.to(device))

            grad_scaler.scale(loss).backward()
            losses.append(loss.detach())
            grad_scaler.step(optimizer)
            grad_scaler.update()
            optimizer.zero_grad()
    losses = torch.tensor(losses).cpu().numpy().tolist()
    val_metric = validate(model, val_loader=val_loader, loss_fn=loss_fn)
    
    return losses, val_metric

In [35]:
model_ens = LinearEnsemble(
                        dataset_size=X_train_for_linear_ensemble_torch.shape[1],
                        n_input_features=X_train_for_linear_ensemble_torch.shape[2],
                        n_output_features=Y_train_ensemble.shape[2],
                    )
model_ens.to(device)


losses_train_independent_linear_models, val_metric = train_and_validate(model_ens, dataloader_train_ensemble, dataloader_test_ensemble, n_epochs=500)
val_metric

100%|██████████| 500/500 [05:24<00:00,  1.54it/s]


4.085072040557861

In [155]:
model_shared = LinearEnsemble(
                        dataset_size=X_train_for_linear_ensemble_torch.shape[1],
                        n_input_features=X_train_for_linear_ensemble_torch.shape[2],
                        n_output_features=Y_train_ensemble.shape[2],
                        shared_weights=True
                    )
model_shared.to(device)


losses_train_shared_linear_models, val_metric = train_and_validate(model_shared, dataloader_train_ensemble, dataloader_test_ensemble, lr=1, n_epochs=500, weight_decay=1e-5)
val_metric

  2%|▏         | 11/500 [00:07<05:16,  1.54it/s]


KeyboardInterrupt: 

In [ ]:
px.line(y = [losses_train_independent_linear_models, losses_train_shared_linear_models])

## Graph model

In [160]:
class GraphModel(nn.Module):
    def __init__(self,
                 dataset_size: int,
                 n_input_features: int,
                 n_output_features: int,
                 edges: np.ndarray = None,
                 shared_weights: bool = False,
                 n_layers: int = 1,
        ):

        super().__init__()
        adj_matrix = self._construct_adjacency_matrix(number_of_nodes=dataset_size, edgelist=edges)
        node_degrees = torch.sum(adj_matrix, 0)
        node_degrees = torch.where(node_degrees == 0, 1, node_degrees)

        reverse_degree_matrix = 1/node_degrees * np.eye(dataset_size)
        self.register_buffer(name="adj_matrix", tensor=adj_matrix.float())
        self.register_buffer(name="reverse_degree_matrix", tensor=reverse_degree_matrix.float())

        self.linear_models = nn.ModuleList(
            [LinearEnsemble(dataset_size=dataset_size, n_input_features=n_input_features, n_output_features=n_output_features, shared_weights=shared_weights)] + [
            LinearEnsemble(dataset_size=dataset_size, n_input_features=2 * n_output_features, n_output_features=n_output_features, shared_weights=shared_weights)
            for _ in range(n_layers-1)
        ])
        self.n_layers = n_layers
        self.final_layer = nn.Linear(in_features=2 * n_output_features, out_features=n_output_features)

    def _graph_aggregation(self, h):
        return torch.cat([self.reverse_degree_matrix @ self.adj_matrix @ h, h], -1)

    @staticmethod
    def _construct_adjacency_matrix(number_of_nodes: int, edgelist: np.ndarray):
        adjacency_matrix = torch.zeros((number_of_nodes, number_of_nodes))
        for u, v in edgelist:
            adjacency_matrix[u, v] = 1.0
        return adjacency_matrix

    def forward(self, x):
        # print(x.shape)
        h = self.linear_models[0](x)
        x = self._graph_aggregation(h)
        # print(h.shape)
        # print(x.shape)
        for layer in self.linear_models[1:]:
            h = layer(x)

            x = self._graph_aggregation(h) + x

        x = self.final_layer(x)
        return x


In [161]:
next(iter(dataloader_train_ensemble))[1].shape

torch.Size([16384, 207, 12])

In [162]:
gnn = GraphModel(dataset_size=X_train_for_linear_ensemble_torch.shape[1],
                 n_input_features=X_train_for_linear_ensemble_torch.shape[2],
                 n_output_features=Y_train_ensemble.shape[2],
                 shared_weights=True,
                 edges=graph,
                 n_layers=2,
)

gnn.to(device)


losses_train_gnn, val_metric = train_and_validate(gnn, dataloader_train_ensemble, dataloader_test_ensemble, lr=0.5, n_epochs=100, weight_decay=1e-5)
val_metric

100%|██████████| 1000/1000 [10:56<00:00,  1.52it/s]


3.6014935970306396

In [ ]:
px.line(y = [losses_train_independent_linear_models[:100], losses_train_shared_linear_models[:100], losses_train_gnn[:100]])